In [ ]:
%matplotlib inline
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os

In [ ]:

file_path = os.path.join('..', 'data', 'raw', 'newsData','raw_analyst_ratings.csv')
news_df=pd.read_csv(file_path)
news_df.head()


In [ ]:
#Descriptive Statistics
#Calculate Headline Lengths
news_df['headline_len'] = news_df['headline'].apply(len)

print("Headline Length Statistics:")
print(news_df['headline_len'].describe())

plt.figure(figsize=(10, 6))
sns.histplot(news_df['headline_len'], bins=50, kde=True, color='blue')
plt.title('Distribution of Headline Lengths')
plt.xlabel('Number of Characters')
plt.ylabel('Frequency')
plt.show()

In [ ]:
publisher_counts = news_df['publisher'].value_counts()

# 2. Get the top 10 most active publishers
top_10_publishers = publisher_counts.head(10)

print("Top 10 Publishers:")
print(top_10_publishers)

plt.figure(figsize=(12, 6))
top_10_publishers.plot(kind='bar', color='teal')
plt.title('Top 10 Most Active News Publishers')
plt.xlabel('Publisher Name')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 1. Convert the 'date' column to datetime objects
# Note: errors='coerce' turns unparseable dates into NaT (Not a Time)
news_df['date'] = pd.to_datetime(news_df['date'], errors='coerce')

# 2. Remove rows where the date couldn't be parsed (if any)
news_df = news_df.dropna(subset=['date'])

# 3. Extract just the date (ignoring the time) to see daily volume
news_df['only_date'] = news_df['date'].dt.date
daily_news_counts = news_df.groupby('only_date').size()

# 4. Visualize the News Volume Spike over time
plt.figure(figsize=(15, 6))
daily_news_counts.plot(kind='line', color='orange')
plt.title('News Publication Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Articles')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Text Analysis (Topic Modeling)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# For CountVectorizer and TF-IDF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# For LDA (Topic Modeling)
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
#cleaning word
import sys
sys.path.append('../scripts') 

from helper import clean_headline

news_df['cleaned_headline'] = news_df['headline'].apply(clean_headline)
news_df.head()

In [ ]:
#CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer
cv = CountVectorizer(max_features=20)  # top 20 most common words

# Fit and transform the cleaned headlines
cv_matrix = cv.fit_transform(news_df['cleaned_headline'])

# Convert to dataframe to see the results
cv_df = pd.DataFrame(cv_matrix.toarray(), columns=cv.get_feature_names_out())

print(cv_df.head())

In [ ]:
# Sum up how many times each word appears
word_counts = cv_df.sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=word_counts.values, y=word_counts.index, palette='Blues_r')
plt.title('Top 20 Most Common Words in Headlines')
plt.xlabel('Count')
plt.ylabel('Words')
plt.tight_layout()
plt.show()

In [ ]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF
tfidf = TfidfVectorizer(max_features=20)  # top 20 important words

# Fit and transform
tfidf_matrix = tfidf.fit_transform(news_df['cleaned_headline'])

# Convert to dataframe
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

print(tfidf_df.head())

In [ ]:
# Average importance score of each word
tfidf_scores = tfidf_df.mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=tfidf_scores.values, y=tfidf_scores.index, palette='Greens_r')
plt.title('Top 20 Most Important Words in Headlines (TF-IDF)')
plt.xlabel('TF-IDF Score')
plt.ylabel('Words')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# Initialize LDA — we'll find 5 topics
lda = LatentDirichletAllocation(n_components=5, random_state=42)

# Fit on the CountVectorizer matrix (LDA works with word counts)
lda.fit(cv_matrix)

# Function to display top words in each topic
def display_topics(model, feature_names, no_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")
        print()

# Display topics
feature_names = cv.get_feature_names_out()
display_topics(lda, feature_names)

import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_indices]
    top_scores = [topic[i] for i in top_indices]

    axes[topic_idx].barh(top_words[::-1], top_scores[::-1], color='purple', alpha=0.7)
    axes[topic_idx].set_title(f'Topic {topic_idx + 1}', fontsize=13)
    axes[topic_idx].set_xlabel('Word Score')

# Hide the extra subplot (we have 5 topics but 6 subplots)
axes[5].set_visible(False)

plt.suptitle('LDA Topics from News Headlines', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Find which topic each headline belongs to
topic_assignments = lda.transform(cv_matrix)
news_df['dominant_topic'] = topic_assignments.argmax(axis=1) + 1  # +1 so topics start at 1

# See the distribution of topics
plt.figure(figsize=(8, 5))
sns.countplot(x='dominant_topic', data=news_df, palette='viridis')
plt.title('Number of Headlines per Topic')
plt.xlabel('Topic')
plt.ylabel('Count')
plt.show()

# Preview headlines per topic
for i in range(1, 6):
    print(f"\n--- Topic {i} Sample Headlines ---")
    print(news_df[news_df['dominant_topic'] == i]['headline'].head(3).to_string())